# Notebook 1: Data Overview and Ingestion

## Objective
This notebook provides an execution-level overview of the Aadhaar enrolment and update datasets used in this study.  
It establishes:

- The project directory structure
- The raw data organisation within the ZIP archive
- The ingestion and standardisation logic used prior to cleaning
- Initial sanity checks on the raw datasets

No cleaning or aggregation is performed in this notebook.  
All transformation logic is implemented in `src/clean.py` and executed in Notebook 2.


## Project Directory Structure

aadhaar_trends_analysis/
│
├── data/
│   ├── raw/
│   │   ├── biometric/
│   │   │   ├── biometric_part_01.csv
│   │   │   ├── biometric_part_02.csv
│   │   │   ├── biometric_part_03.csv
│   │   │   └── biometric_part_04.csv
│   │   │
│   │   ├── demographic/
│   │   │   ├── demographic_part_01.csv
│   │   │   ├── demographic_part_02.csv
│   │   │   └── demographic_part_03.csv
│   │   │
│   │   └── enrolment/
│   │       ├── enrolment_part_01.csv
│   │       ├── enrolment_part_02.csv
│   │       └── enrolment_part_03.csv
│   │
│   └── processed/
│       ├── biometric_clean.csv
│       ├── demographic_clean.csv
│       └── enrolment_clean.csv
│
├── notebooks/                    ← EXPLORATORY & EXPLANATORY (MANDATORY)
│   ├── Notebook_01_Data_Overview.ipynb
│   ├── Notebook_02_Cleaning_Preprocessing.ipynb
│   ├── Notebook_03_Univariate_Bivariate_Analysis.ipynb
│   ├── Notebook_04_Stress_Risk_Modeling.ipynb
│   ├── Notebook_05_Integrated_Insights.ipynb
│   └── Notebook_06_Predictive_Extension.ipynb
│
├── src/                          ← CORE ANALYSIS & PIPELINE
│   ├── clean.py
│   │
│   ├── analyse_enrolment.py
│   ├── analyse_biometric.py
│   ├── analyse_demographic.py
│   │
│   ├── integrated_analysis.py
│   │
│   ├── system_stress_index.py
│   ├── state_dominance_analysis.py
│   ├── failure_mode_tagging.py
│   ├── extreme_scale_analysis.py
│   ├── decision_analysis.py
│   ├── ml_risk_intelligence.py
│   │
│   └── run_pipeline.py
│
├── outputs/
│   ├── figures/
│   │   ├── enrolment_trends.png
│   │   ├── biometric_load.png
│   │   └── risk_zone_distribution.png
│   │
│   └── tables/
│       ├── system_stress_index.csv
│       ├── state_risk_score.csv
│       └── decision_table.csv
│
├── app/                          ← OPTIONAL PRESENTATION / DASHBOARD
│   └── app.py
│
├── README.md
├── requirements.txt
└── .gitignore

All file paths referenced in this notebook and subsequent notebooks strictly follow this structure.


In [1]:
import os
import glob
import pandas as pd


In [2]:
# Path configuration (must match clean.py)

RAW_BASE = "data/raw"

ENROLMENT_PATH = os.path.join(RAW_BASE, "enrolment")
BIOMETRIC_PATH = os.path.join(RAW_BASE, "biometric")
DEMOGRAPHIC_PATH = os.path.join(RAW_BASE, "demographic")

ENROLMENT_PATH, BIOMETRIC_PATH, DEMOGRAPHIC_PATH


('data/raw\\enrolment', 'data/raw\\biometric', 'data/raw\\demographic')

In [3]:
def list_csv_files(folder):
    return [os.path.basename(f) for f in glob.glob(os.path.join(folder, "*.csv"))]

print("Enrolment CSVs :", list_csv_files(ENROLMENT_PATH))
print("Biometric CSVs :", list_csv_files(BIOMETRIC_PATH))
print("Demographic CSVs:", list_csv_files(DEMOGRAPHIC_PATH))


Enrolment CSVs : []
Biometric CSVs : []
Demographic CSVs: []


## Note on Raw File Naming

The raw CSV files in this project have been renamed and grouped into domain-specific folders
(`enrolment`, `biometric`, `demographic`) for the following reasons:

- Improved readability and maintainability
- Consistent ingestion using wildcard loading (`glob`)
- Clear separation of operational domains

This renaming is strictly nominal.  
No data values, columns, or semantics were altered during renaming.


In [4]:
# ================================
# CELL 1: Robust raw enrolment load
# ================================

import os
import glob
import pandas as pd

# ---- Step 1: Ensure correct project root ----
# This assumes the notebook is inside aadhar_analysis/
# If not, we dynamically fix the path

PROJECT_ROOT = os.getcwd()

# If 'data' folder is not found, try moving one level up
if not os.path.exists(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))

RAW_BASE = os.path.join(PROJECT_ROOT, "data", "raw")
ENROLMENT_PATH = os.path.join(RAW_BASE, "enrolment")

# ---- Step 2: Validate directory ----
print("PROJECT_ROOT :", PROJECT_ROOT)
print("ENROLMENT_PATH:", ENROLMENT_PATH)

if not os.path.exists(ENROLMENT_PATH):
    raise FileNotFoundError(f"Enrolment folder not found at: {ENROLMENT_PATH}")

# ---- Step 3: Locate CSV files ----
enrolment_files = glob.glob(os.path.join(ENROLMENT_PATH, "*.csv"))

print("Found enrolment files:", enrolment_files)

if len(enrolment_files) == 0:
    raise FileNotFoundError("No enrolment CSV files found")

# ---- Step 4: Load first file safely ----
sample_file = enrolment_files[0]
raw_enrolment_sample = pd.read_csv(sample_file)

raw_enrolment_sample.head()


PROJECT_ROOT : d:\aadhaar_trends_analysis\notebooks
ENROLMENT_PATH: d:\aadhaar_trends_analysis\notebooks\data\raw\enrolment


FileNotFoundError: Enrolment folder not found at: d:\aadhaar_trends_analysis\notebooks\data\raw\enrolment

In [ ]:
# Inspect enrolment raw data schema
raw_enrolment_sample.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   date            500000 non-null  object
 1   state           500000 non-null  object
 2   district        500000 non-null  object
 3   pincode         500000 non-null  int64 
 4   age_0_5         500000 non-null  int64 
 5   age_5_17        500000 non-null  int64 
 6   age_18_greater  500000 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 26.7+ MB


## Column Normalisation Strategy

As implemented in `clean.py`, all raw datasets undergo column standardisation:

- Stripping whitespace
- Converting column names to lowercase

This ensures consistency across datasets originating from different reporting formats.


In [ ]:
# Preview normalised column names (do not overwrite)
raw_enrolment_sample.columns.str.strip().str.lower().tolist()


['date',
 'state',
 'district',
 'pincode',
 'age_0_5',
 'age_5_17',
 'age_18_greater']

In [ ]:
# ================================
# CELL 4: Robust raw biometric load
# ================================

import os
import glob
import pandas as pd

# ---- Step 1: Reuse PROJECT_ROOT logic (same as Cell 1) ----
PROJECT_ROOT = os.getcwd()

if not os.path.exists(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))

RAW_BASE = os.path.join(PROJECT_ROOT, "data", "raw")
BIOMETRIC_PATH = os.path.join(RAW_BASE, "biometric")

# ---- Step 2: Validate biometric directory ----
print("PROJECT_ROOT   :", PROJECT_ROOT)
print("BIOMETRIC_PATH :", BIOMETRIC_PATH)

if not os.path.exists(BIOMETRIC_PATH):
    raise FileNotFoundError(f"Biometric folder not found at: {BIOMETRIC_PATH}")

# ---- Step 3: Locate biometric CSV files ----
biometric_files = glob.glob(os.path.join(BIOMETRIC_PATH, "*.csv"))

print("Found biometric files:", biometric_files)

if len(biometric_files) == 0:
    raise FileNotFoundError("No biometric CSV files found")

# ---- Step 4: Load first biometric file safely ----
sample_bio = biometric_files[0]
raw_bio_sample = pd.read_csv(sample_bio)

raw_bio_sample.head(), raw_bio_sample.columns


PROJECT_ROOT   : d:\aadhaar_trends_analysis
BIOMETRIC_PATH : d:\aadhaar_trends_analysis\data\raw\biometric
Found biometric files: ['d:\\aadhaar_trends_analysis\\data\\raw\\biometric\\biometric_part_01.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\biometric\\biometric_part_02.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\biometric\\biometric_part_03.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\biometric\\biometric_part_04.csv']


(         date              state      district  pincode  bio_age_5_17  \
 0  01-03-2025            Haryana  Mahendragarh   123029           280   
 1  01-03-2025              Bihar     Madhepura   852121           144   
 2  01-03-2025  Jammu and Kashmir         Punch   185101           643   
 3  01-03-2025              Bihar       Bhojpur   802158           256   
 4  01-03-2025         Tamil Nadu       Madurai   625514           271   
 
    bio_age_17_  
 0          577  
 1          369  
 2         1091  
 3          980  
 4          815  ,
 Index(['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_'], dtype='object'))

In [ ]:
# Inspect biometric raw data schema
raw_bio_sample.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   date          500000 non-null  object
 1   state         500000 non-null  object
 2   district      500000 non-null  object
 3   pincode       500000 non-null  int64 
 4   bio_age_5_17  500000 non-null  int64 
 5   bio_age_17_   500000 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 22.9+ MB


In [ ]:
# ================================
# CELL 6: Robust raw demographic load
# ================================

import os
import glob
import pandas as pd

# ---- Step 1: Resolve project root safely ----
PROJECT_ROOT = os.getcwd()

# If notebook is inside a subfolder (e.g., notebooks/), move up
if not os.path.exists(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))

RAW_BASE = os.path.join(PROJECT_ROOT, "data", "raw")
DEMOGRAPHIC_PATH = os.path.join(RAW_BASE, "demographic")

# ---- Step 2: Validate demographic directory ----
print("PROJECT_ROOT      :", PROJECT_ROOT)
print("DEMOGRAPHIC_PATH  :", DEMOGRAPHIC_PATH)

if not os.path.exists(DEMOGRAPHIC_PATH):
    raise FileNotFoundError(
        f"Demographic folder not found.\nExpected at: {DEMOGRAPHIC_PATH}"
    )

# ---- Step 3: Locate demographic CSV files ----
demographic_files = glob.glob(os.path.join(DEMOGRAPHIC_PATH, "*.csv"))

print("Found demographic files:", demographic_files)

if len(demographic_files) == 0:
    raise FileNotFoundError(
        f"No demographic CSV files found in {DEMOGRAPHIC_PATH}"
    )

# ---- Step 4: Load first demographic file safely ----
sample_demo = demographic_files[0]
raw_demo_sample = pd.read_csv(sample_demo)

raw_demo_sample.head(), raw_demo_sample.columns


PROJECT_ROOT      : d:\aadhaar_trends_analysis
DEMOGRAPHIC_PATH  : d:\aadhaar_trends_analysis\data\raw\demographic
Found demographic files: ['d:\\aadhaar_trends_analysis\\data\\raw\\demographic\\demographic_part_01.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\demographic\\demographic_part_02.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\demographic\\demographic_part_03.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\demographic\\demographic_part_04.csv', 'd:\\aadhaar_trends_analysis\\data\\raw\\demographic\\demographic_part_05.csv']


(         date           state    district  pincode  demo_age_5_17  \
 0  01-03-2025   Uttar Pradesh   Gorakhpur   273213             49   
 1  01-03-2025  Andhra Pradesh    Chittoor   517132             22   
 2  01-03-2025         Gujarat      Rajkot   360006             65   
 3  01-03-2025  Andhra Pradesh  Srikakulam   532484             24   
 4  01-03-2025       Rajasthan     Udaipur   313801             45   
 
    demo_age_17_  
 0           529  
 1           375  
 2           765  
 3           314  
 4           785  ,
 Index(['date', 'state', 'district', 'pincode', 'demo_age_5_17',
        'demo_age_17_'],
       dtype='object'))

## Alignment With Cleaning Pipeline

The inspection above confirms that:

- All datasets contain the required geographic and temporal fields
- Column naming variations exist across sources
- Age-segmented metrics require standardisation

These observations directly motivate the cleaning and consolidation logic implemented in `src/clean.py`,
which is executed in the next notebook.

No assumptions or transformations are performed manually in this notebook.


## Summary of Notebook 1

This notebook established:

- The raw data organisation within the ZIP archive
- The ingestion strategy used across all datasets
- Schema variability motivating standardised cleaning
- Consistency between project structure and `clean.py`

The next notebook executes the full cleaning and consolidation pipeline using the defined scripts.
